# Stage 3: CBF-Integrated Training
## 核心思路:
1. UNet生成初始速度场
2. CBF计算安全速度修正
3. 将修正后的速度作为新的训练目标
4. 迭代训练,让UNet学会生成安全的速度

In [3]:
# Cell 0: 生成训练数据 (与之前相同)

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

def sample_point_in_circle(center, radius):
    angle = np.random.uniform(0, 2 * np.pi)
    r = radius * np.sqrt(np.random.uniform(0, 1))
    return center + r * np.array([np.cos(angle), np.sin(angle)])

def generate_bezier_curve(P0, P1, P2, P3, num_points=100):
    t = np.linspace(0, 1, num_points).reshape(-1, 1)
    B_t = (1 - t)**3 * P0 + 3 * (1 - t)**2 * t * P1 + 3 * (1 - t) * t**2 * P2 + t**3 * P3
    return B_t

baseline_start = np.array([-1, -1])
baseline_end   = np.array([ 1,  1])
radius = 0.2
dominant_endpoint = "start"

if dominant_endpoint == "start":
    perturbation_scale_P1 = 0.7
    perturbation_scale_P2 = 0.01
elif dominant_endpoint == "end":
    perturbation_scale_P1 = 0.1
    perturbation_scale_P2 = 0.3
else:
    perturbation_scale_P1 = perturbation_scale_P2 = 0.2

num_trajectories = 8000
trajectories = []

print(f"生成 {num_trajectories} 条Bezier轨迹...")
for i in range(num_trajectories):
    P0 = sample_point_in_circle(baseline_start, radius)
    P3 = sample_point_in_circle(baseline_end, radius)
    
    base_vector = P3 - P0
    perp_vector = np.array([-base_vector[1], base_vector[0]])
    perp_vector = perp_vector / (np.linalg.norm(perp_vector) + 1e-8)
  
    P1 = P0 + base_vector / 3.0
    P2 = P0 + 2 * base_vector / 3.0
    
    P1 = P1 + np.random.uniform(-perturbation_scale_P1, perturbation_scale_P1) * perp_vector
    P2 = P2 + np.random.uniform(-perturbation_scale_P2, perturbation_scale_P2) * perp_vector
    
    curve = generate_bezier_curve(P0, P1, P2, P3, num_points=100)
    curve = curve.T
    trajectories.append(curve)

trajectories = np.array(trajectories)
print(f"✓ 生成完成: {trajectories.shape}")

生成 8000 条Bezier轨迹...
✓ 生成完成: (8000, 2, 100)


In [4]:
# Cell 1: 定义UNet模型 (与之前相同)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

class TrajectoryDataset(Dataset):
    def __init__(self, trajectories):
        self.trajectories = trajectories
    def __len__(self):
        return len(self.trajectories)
    def __getitem__(self, idx):
        return torch.tensor(self.trajectories[idx], dtype=torch.float32)

class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        if t.dim() == 0:
            t = t.unsqueeze(0)
        if t.dim() == 1:
            t = t.unsqueeze(-1)
        emb = t * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class ResBlock1D(nn.Module):
    def __init__(self, channels, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Sequential(Mish(), nn.Linear(time_emb_dim, channels))
        self.block = nn.Sequential(
            nn.GroupNorm(8, channels), Mish(),
            nn.Conv1d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels), Mish(),
            nn.Conv1d(channels, channels, 3, padding=1),
        )
    def forward(self, x, t_emb):
        h = self.block(x)
        if t_emb.shape[0] == 1 and x.shape[0] > 1:
            t_emb = t_emb.expand(x.shape[0], -1)
        h = h + self.time_mlp(t_emb)[:, :, None]
        return x + h

class SimpleUNet1D(nn.Module):
    def __init__(self, time_emb_dim=128, hidden_dim=128):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        self.hidden_dim = hidden_dim
        
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 4), Mish(),
            nn.Linear(time_emb_dim * 4, time_emb_dim)
        )
        
        self.conv_in = nn.Conv1d(2, hidden_dim, 3, padding=1)
        
        self.down1_block = nn.ModuleList([ResBlock1D(hidden_dim, time_emb_dim), ResBlock1D(hidden_dim, time_emb_dim)])
        self.down1_sample = nn.Conv1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        
        self.down2_block = nn.ModuleList([ResBlock1D(hidden_dim, time_emb_dim), ResBlock1D(hidden_dim, time_emb_dim)])
        self.down2_sample = nn.Conv1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        
        self.mid_block = nn.ModuleList([ResBlock1D(hidden_dim, time_emb_dim), ResBlock1D(hidden_dim, time_emb_dim)])
        
        self.up2_sample = nn.ConvTranspose1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        self.up2_block = nn.ModuleList([ResBlock1D(hidden_dim * 2, time_emb_dim), ResBlock1D(hidden_dim * 2, time_emb_dim)])
        self.up2_conv = nn.Conv1d(hidden_dim * 2, hidden_dim, 1)
        
        self.up1_sample = nn.ConvTranspose1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        self.up1_block = nn.ModuleList([ResBlock1D(hidden_dim * 2, time_emb_dim), ResBlock1D(hidden_dim * 2, time_emb_dim)])
        self.up1_conv = nn.Conv1d(hidden_dim * 2, hidden_dim, 1)
        
        self.conv_out = nn.Conv1d(hidden_dim, 2, 3, padding=1)
    
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        
        x = self.conv_in(x)
        
        # Downsampling
        d1 = x
        for block in self.down1_block:
            d1 = block(d1, t_emb)
        d1_down = self.down1_sample(d1)
        
        d2 = d1_down
        for block in self.down2_block:
            d2 = block(d2, t_emb)
        d2_down = self.down2_sample(d2)
        
        # Middle
        m = d2_down
        for block in self.mid_block:
            m = block(m, t_emb)
        
        # Upsampling
        u2 = self.up2_sample(m)
        u2 = torch.cat([u2, d2], dim=1)
        for block in self.up2_block:
            u2 = block(u2, t_emb)
        u2 = self.up2_conv(u2)
        
        u1 = self.up1_sample(u2)
        u1 = torch.cat([u1, d1], dim=1)
        for block in self.up1_block:
            u1 = block(u1, t_emb)
        u1 = self.up1_conv(u1)
        
        out = self.conv_out(u1)
        return out

print("✓ 模型定义完成")

使用设备: cuda
✓ 模型定义完成


In [6]:
# Cell 2: CBF速度计算模块
# 这是关键:给定当前状态和UNet预测的速度,计算CBF修正后的安全速度

import cvxpy as cp

def compute_cbf_safe_velocity(x, v_nominal, obstacles, alpha=0.5, dt=0.01):
    """
    使用CBF计算安全速度修正
    
    Args:
        x: 当前位置 (2,)
        v_nominal: UNet预测的名义速度 (2,)
        obstacles: 障碍物列表
        alpha: CBF参数
        dt: 时间步长
    
    Returns:
        v_safe: CBF修正后的安全速度 (2,)
        correction: 速度修正量 v_safe - v_nominal
    """
    # 优化变量: 速度
    v = cp.Variable(2)
    
    # 目标: 最小化与名义速度的偏差
    objective = cp.Minimize(cp.sum_squares(v - v_nominal))
    
    # 约束: CBF条件
    constraints = []
    
    for obs in obstacles:
        center = obs['center']
        radius = obs['radius']
        safety_margin = obs.get('safety_margin', 0.2)
        
        # CBF: h(x) = ||x - center||^2 - (radius + safety_margin)^2
        dx = x - center
        h = np.dot(dx, dx) - (radius + safety_margin)**2
        
        # CBF约束: dh/dt + alpha * h >= 0
        # dh/dt = 2 * dx^T * v
        constraints.append(2 * dx @ v + alpha * h >= 0)
    
    # 速度限制
    max_vel = 2.5
    constraints.append(cp.norm(v, 2) <= max_vel)
    
    # 求解QP
    problem = cp.Problem(objective, constraints)
    
    try:
        problem.solve(solver=cp.OSQP, warm_start=True, verbose=False)
        
        if problem.status in ['optimal', 'optimal_inaccurate']:
            v_safe = v.value
            correction = v_safe - v_nominal
            return v_safe, correction
        else:
            # 如果求解失败,返回原始速度
            return v_nominal, np.zeros(2)
    except:
        return v_nominal, np.zeros(2)

# 批量处理整条轨迹
def compute_cbf_corrected_trajectory(traj, v_field, obstacles, alpha=0.5, dt=0.01):
    """
    对整条轨迹的速度场进行CBF修正
    
    Args:
        traj: 轨迹 (2, T)
        v_field: UNet预测的速度场 (2, T)
        obstacles: 障碍物列表
    
    Returns:
        v_corrected: 修正后的速度场 (2, T)
        corrections: 速度修正量 (2, T)
    """
    T = traj.shape[1]
    v_corrected = np.zeros_like(v_field)
    corrections = np.zeros_like(v_field)
    
    for t in range(T):
        x_t = traj[:, t]
        v_nominal = v_field[:, t]
        
        v_safe, corr = compute_cbf_safe_velocity(x_t, v_nominal, obstacles, alpha, dt)
        
        v_corrected[:, t] = v_safe
        corrections[:, t] = corr
    
    return v_corrected, corrections

print("✓ CBF速度计算模块定义完成")

ModuleNotFoundError: No module named 'cvxpy'

In [ ]:
# Cell 3: 定义障碍物

obstacles = [
    {'center': np.array([0.0, 0.0]), 'radius': 0.3, 'safety_margin': 0.2},
    {'center': np.array([0.5, -0.3]), 'radius': 0.2, 'safety_margin': 0.2},
    {'center': np.array([-0.4, 0.5]), 'radius': 0.25, 'safety_margin': 0.2},
]

print(f"✓ 定义了 {len(obstacles)} 个障碍物")

In [ ]:
# Cell 4: CBF-Integrated训练循环
# 这是核心创新:在训练过程中集成CBF反馈

from fmtorch.scheduler import CondOTScheduler
from fmtorch.path import AffinePath

def train_with_cbf_integration(
    model, 
    train_loader, 
    obstacles,
    num_epochs=50, 
    lr=1e-4,
    cbf_weight=0.3,  # CBF修正的权重
    device='cuda'
):
    """
    CBF集成训练:
    1. 标准Flow Matching loss
    2. + CBF修正的速度作为额外监督
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler_fm = CondOTScheduler(sigma=0.0)
    
    model.train()
    history = {'loss': [], 'fm_loss': [], 'cbf_loss': []}
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_fm_loss = 0.0
        epoch_cbf_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, z1 in enumerate(pbar):
            z1 = z1.to(device)  # (B, 2, T)
            B, C, T = z1.shape
            
            # 采样时间和噪声
            t = torch.rand(B, device=device) * 0.99 + 0.005
            z0 = torch.randn_like(z1)
            
            # Flow Matching路径
            path = AffinePath(scheduler_fm)
            zt = path.sample(t[:, None, None], z0, z1)
            
            # 标准Flow Matching目标: v_t = z1 - noise
            target_velocity_fm = path.dx_dt(t[:, None, None], z0, z1)
            
            # === 关键创新: CBF修正 ===
            # 对于batch中的每个样本,计算CBF修正后的目标速度
            target_velocity_cbf = torch.zeros_like(target_velocity_fm)
            
            for b in range(B):
                # 当前位置和FM速度
                traj_b = zt[b].cpu().detach().numpy()  # (2, T)
                v_fm_b = target_velocity_fm[b].cpu().detach().numpy()  # (2, T)
                
                # 使用CBF计算修正后的速度
                v_corrected, _ = compute_cbf_corrected_trajectory(
                    traj_b, v_fm_b, obstacles, alpha=0.5
                )
                
                target_velocity_cbf[b] = torch.tensor(v_corrected, dtype=torch.float32, device=device)
            
            # UNet预测
            pred_velocity = model(zt, t)
            
            # 混合loss:
            # 1. 标准FM loss: 让预测接近原始目标
            loss_fm = F.mse_loss(pred_velocity, target_velocity_fm)
            
            # 2. CBF loss: 让预测接近CBF修正后的目标
            loss_cbf = F.mse_loss(pred_velocity, target_velocity_cbf)
            
            # 总loss
            loss = (1 - cbf_weight) * loss_fm + cbf_weight * loss_cbf
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_fm_loss += loss_fm.item()
            epoch_cbf_loss += loss_cbf.item()
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'fm': f"{loss_fm.item():.4f}",
                'cbf': f"{loss_cbf.item():.4f}"
            })
        
        # 记录历史
        avg_loss = epoch_loss / len(train_loader)
        avg_fm = epoch_fm_loss / len(train_loader)
        avg_cbf = epoch_cbf_loss / len(train_loader)
        
        history['loss'].append(avg_loss)
        history['fm_loss'].append(avg_fm)
        history['cbf_loss'].append(avg_cbf)
        
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, FM={avg_fm:.4f}, CBF={avg_cbf:.4f}")
    
    return history

print("✓ CBF-Integrated训练函数定义完成")

In [ ]:
# Cell 5: 初始化模型和数据

model = SimpleUNet1D(time_emb_dim=128, hidden_dim=128).to(device)
dataset = TrajectoryDataset(trajectories)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"✓ 模型参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"✓ 训练集大小: {len(dataset)}")

In [ ]:
# Cell 6: 开始CBF-Integrated训练

print("=" * 70)
print("开始CBF-Integrated训练")
print("=" * 70)

history = train_with_cbf_integration(
    model=model,
    train_loader=train_loader,
    obstacles=obstacles,
    num_epochs=30,
    lr=1e-4,
    cbf_weight=0.3,  # 30%的权重给CBF修正
    device=device
)

print("\n✓ 训练完成!")

In [ ]:
# Cell 7: 可视化训练历史

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss曲线
axes[0].plot(history['loss'], label='Total Loss', linewidth=2)
axes[0].plot(history['fm_loss'], label='FM Loss', linewidth=2, alpha=0.7)
axes[0].plot(history['cbf_loss'], label='CBF Loss', linewidth=2, alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss ratio
fm_ratio = np.array(history['fm_loss']) / (np.array(history['loss']) + 1e-8)
cbf_ratio = np.array(history['cbf_loss']) / (np.array(history['loss']) + 1e-8)
axes[1].plot(fm_ratio, label='FM Contribution', linewidth=2)
axes[1].plot(cbf_ratio, label='CBF Contribution', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss Ratio')
axes[1].set_title('Loss Component Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: 测试生成轨迹

from fmtorch.solver import ODESolver
from fmtorch.utils import ModelWrapper

def sample_trajectories(model, num_samples=10, num_steps=100, device='cuda'):
    model.eval()
    
    # 初始噪声
    z0 = torch.randn(num_samples, 2, 100).to(device)
    
    # 包装模型
    wrapped_model = ModelWrapper(model)
    
    # ODE求解
    solver = ODESolver(method='dopri5')
    z1 = solver.sample(wrapped_model, z0, step_size=num_steps)
    
    return z1.cpu().detach().numpy()

print("生成测试轨迹...")
test_trajs = sample_trajectories(model, num_samples=10, device=device)
print(f"✓ 生成完成: {test_trajs.shape}")

In [ ]:
# Cell 9: 评估碰撞检测

def check_collision(traj, obstacles):
    """检查轨迹是否与障碍物碰撞"""
    collisions = 0
    for t in range(traj.shape[1]):
        x = traj[:, t]
        for obs in obstacles:
            dist = np.linalg.norm(x - obs['center']) - obs['radius']
            if dist < 0:
                collisions += 1
    return collisions

def check_safety_margin(traj, obstacles, safety_margin=0.2):
    """检查轨迹是否违反安全边界"""
    violations = 0
    for t in range(traj.shape[1]):
        x = traj[:, t]
        for obs in obstacles:
            dist = np.linalg.norm(x - obs['center']) - (obs['radius'] + safety_margin)
            if dist < 0:
                violations += 1
    return violations

print("安全性评估:")
print("=" * 70)

total_collisions = 0
total_violations = 0

for i in range(len(test_trajs)):
    traj = test_trajs[i]
    col = check_collision(traj, obstacles)
    viol = check_safety_margin(traj, obstacles)
    
    total_collisions += col
    total_violations += viol
    
    if col > 0:
        print(f"轨迹 {i+1}: {col} 碰撞点")

print("=" * 70)
print(f"总碰撞点: {total_collisions}")
print(f"总安全违规: {total_violations}")

if total_collisions == 0:
    print("\n🎉 完美! 所有轨迹都是安全的!")
elif total_collisions < len(test_trajs) * 5:  # 平均每条轨迹<5个碰撞点
    print("\n✓ 良好! 碰撞率较低")
else:
    print("\n⚠️ 需要进一步优化")

In [ ]:
# Cell 10: 可视化结果

fig, axes = plt.subplots(2, 5, figsize=(25, 10))

for i in range(min(10, len(test_trajs))):
    row = i // 5
    col = i % 5
    ax = axes[row, col]
    
    # 绘制障碍物
    for obs in obstacles:
        circle = plt.Circle(obs['center'], obs['radius'], 
                           color='red', alpha=0.3, zorder=1)
        ax.add_patch(circle)
        circle_s = plt.Circle(obs['center'], obs['radius'] + 0.2,
                             color='orange', fill=False, linestyle='--',
                             alpha=0.6, linewidth=2, zorder=1)
        ax.add_patch(circle_s)
    
    # 绘制轨迹
    traj = test_trajs[i]
    ax.plot(traj[0], traj[1], 'b-', linewidth=2, zorder=2)
    ax.scatter(traj[0, 0], traj[1, 0], c='green', s=100, 
              marker='o', edgecolors='black', zorder=3)
    ax.scatter(traj[0, -1], traj[1, -1], c='red', s=100,
              marker='*', edgecolors='black', zorder=3)
    
    ax.set_title(f'Trajectory {i+1}')
    ax.axis('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ 可视化完成")

## 总结

### 关键改进:
1. **训练时集成CBF**: 不再是后处理,而是在训练过程中就让模型学习安全约束
2. **混合loss**: 同时优化Flow Matching准确性和CBF安全性
3. **端到端学习**: 模型直接学会生成安全的速度场

### 参数调节:
- `cbf_weight`: 控制CBF修正的强度(建议0.2-0.4)
- `alpha`: CBF参数,控制约束的严格程度
- 训练epochs: 根据收敛情况调整

### 下一步:
1. 实验不同的cbf_weight值
2. 可以添加在线微调:在inference时进一步使用CBF修正
3. 考虑curriculum learning: 逐步增加CBF权重